# Financial Forecasting Models and Evaluation

## 1. Introduction
This notebook builds a production-grade forecasting pipeline for financial time series data. The objective is to predict **returns** derived from the close price using a time-based train/test split, multiple models, and rigorous evaluation. Returns are chosen to stabilize the scale and focus the models on relative changes rather than raw price levels.

## 2. Data Loading & Validation

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_PATH = Path("data/raw/market_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected data file not found: {DATA_PATH.resolve()}")

df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

In [ ]:
print("Shape:", df_raw.shape)
df_raw.info()

In [ ]:
# Standardize column names if needed
df_raw.columns = [c.strip() for c in df_raw.columns]

rename_map = {
    "TimeStamp": "timestamp",
    "OpenPrice": "open",
    "HighPrice": "high",
    "LowPrice": "low",
    "ClosePrice": "close",
    "Volume": "volume",
}

# Case-insensitive rename
rename_map_ci = {k.lower(): v for k, v in rename_map.items()}
df_raw = df_raw.rename(
    columns={c: rename_map_ci[c.lower()] for c in df_raw.columns if c.lower() in rename_map_ci}
)

required_cols = ["timestamp", "open", "high", "low", "close", "volume"]
missing_required = [c for c in required_cols if c not in df_raw.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Parse timestamps and sort
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], errors="coerce", utc=True)
df_raw = df_raw.dropna(subset=["timestamp"]).sort_values("timestamp")

print("Missing values by column:\n", df_raw.isna().sum())
df_raw.describe(include="all")

## 3. Feature Engineering (if not already done)

In [ ]:
def ensure_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure core forecasting features are present. If missing, create them.
    Features are based on historical close values and returns.
    """
    df = df.copy()

    if "returns" not in df.columns:
        df["returns"] = df["close"].pct_change()
    if "ma7" not in df.columns:
        df["ma7"] = df["close"].rolling(window=7).mean()
    if "ma30" not in df.columns:
        df["ma30"] = df["close"].rolling(window=30).mean()
    if "volatility_30" not in df.columns:
        df["volatility_30"] = df["returns"].rolling(window=30).std()

    for lag in [1, 2, 3]:
        col = f"lag_{lag}"
        if col not in df.columns:
            df[col] = df["returns"].shift(lag)

    return df


df_feat = ensure_features(df_raw)
df_feat.head()

## 4. Train/Test Split (Time-Based)

In [ ]:
# Avoid look-ahead bias: shift contemporaneous features by 1 step
feature_cols = ["returns", "ma7", "ma30", "volatility_30", "volume", "lag_1", "lag_2", "lag_3"]
shift_cols = ["returns", "ma7", "ma30", "volatility_30", "volume"]

df_model = df_feat.copy()
df_model[shift_cols] = df_model[shift_cols].shift(1)

# Target: returns
target_col = "returns"

# Drop rows with NaNs introduced by rolling windows and shifts
df_model = df_model.dropna().reset_index(drop=True)

X = df_model[feature_cols]
y = df_model[target_col]

split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## 5. Baseline Model (Naive)

In [ ]:
def rmse(y_true, y_pred) -> float:
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate(y_true, y_pred) -> dict:
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
    }


def naive_previous_value_forecast(y_train: pd.Series, y_test: pd.Series) -> pd.Series:
    """
    Naive forecast: prediction = previous observed value.
    The first prediction in the test set uses the last train value.
    """
    y_pred = y_test.shift(1)
    y_pred.iloc[0] = y_train.iloc[-1]
    return y_pred


y_pred_naive = naive_previous_value_forecast(y_train, y_test)
baseline_metrics = evaluate(y_test, y_pred_naive)
baseline_metrics

## 6. Machine Learning Models

In [ ]:
# Model 1: Linear Regression (with scaling)
linreg_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)
linreg_model.fit(X_train, y_train)
y_pred_linreg = linreg_model.predict(X_test)
linreg_metrics = evaluate(y_test, y_pred_linreg)

linreg_metrics

In [ ]:
# Model 2: Random Forest
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_metrics = evaluate(y_test, y_pred_rf)

rf_metrics

### Optional: XGBoost (if available)

In [ ]:
xgb_metrics = None
xgb_model = None

try:
    from xgboost import XGBRegressor

    xgb_model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    xgb_metrics = evaluate(y_test, y_pred_xgb)
except Exception as exc:
    print("XGBoost not available or failed to run:", exc)

xgb_metrics

## 7. (Optional Advanced) Deep Learning Model (LSTM)
This section is optional and will only run if TensorFlow is available.

In [ ]:
RUN_LSTM = False

lstm_metrics = None

if RUN_LSTM:
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models

        # Prepare sequences
        seq_len = 20
        feature_array = X.values
        target_array = y.values

        X_seq, y_seq = [], []
        for i in range(seq_len, len(feature_array)):
            X_seq.append(feature_array[i - seq_len : i])
            y_seq.append(target_array[i])

        X_seq = np.array(X_seq)
        y_seq = np.array(y_seq)

        split_seq = int(len(X_seq) * 0.8)
        X_train_seq, X_test_seq = X_seq[:split_seq], X_seq[split_seq:]
        y_train_seq, y_test_seq = y_seq[:split_seq], y_seq[split_seq:]

        # LSTM model
        lstm = models.Sequential([
            layers.Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
            layers.LSTM(64, return_sequences=False),
            layers.Dense(1),
        ])

        lstm.compile(optimizer="adam", loss="mse")
        lstm.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, verbose=0)

        y_pred_lstm = lstm.predict(X_test_seq).flatten()
        lstm_metrics = evaluate(pd.Series(y_test_seq), pd.Series(y_pred_lstm))

    except Exception as exc:
        print("LSTM not available or failed to run:", exc)

lstm_metrics

## 8. Model Comparison

In [ ]:
results = [
    {"model": "Naive", "rmse": baseline_metrics["rmse"], "mae": baseline_metrics["mae"], "object": None},
    {"model": "Linear Regression", "rmse": linreg_metrics["rmse"], "mae": linreg_metrics["mae"], "object": linreg_model},
    {"model": "Random Forest", "rmse": rf_metrics["rmse"], "mae": rf_metrics["mae"], "object": rf_model},
]

if xgb_metrics is not None:
    results.append({"model": "XGBoost", "rmse": xgb_metrics["rmse"], "mae": xgb_metrics["mae"], "object": xgb_model})

if lstm_metrics is not None:
    results.append({"model": "LSTM", "rmse": lstm_metrics["rmse"], "mae": lstm_metrics["mae"], "object": None})

results_df = pd.DataFrame(results).sort_values("rmse")
results_df.reset_index(drop=True)

## 9. Visualization

In [ ]:
best_model_row = results_df.iloc[0]
best_model_name = best_model_row["model"]

if best_model_name == "Linear Regression":
    y_pred_best = y_pred_linreg
elif best_model_name == "Random Forest":
    y_pred_best = y_pred_rf
elif best_model_name == "XGBoost":
    y_pred_best = y_pred_xgb
else:
    y_pred_best = y_pred_naive

plt.figure(figsize=(12, 5))
plt.plot(y_test.index, y_test.values, label="Actual", color="#1b4965")
plt.plot(y_test.index, y_pred_best, label=f"Predicted ({best_model_name})", color="#f29e4c")
plt.title("Actual vs Predicted Returns (Test Set)")
plt.xlabel("Time Index")
plt.ylabel("Returns")
plt.legend()
plt.tight_layout()

In [ ]:
# Zoom on the last 20% of the test period
zoom_size = max(20, int(len(y_test) * 0.2))
zoom_idx = y_test.index[-zoom_size:]

plt.figure(figsize=(12, 5))
plt.plot(zoom_idx, y_test.loc[zoom_idx], label="Actual", color="#1b4965")
plt.plot(zoom_idx, pd.Series(y_pred_best, index=y_test.index).loc[zoom_idx], label=f"Predicted ({best_model_name})", color="#f29e4c")
plt.title("Actual vs Predicted Returns (Zoomed Test Window)")
plt.xlabel("Time Index")
plt.ylabel("Returns")
plt.legend()
plt.tight_layout()

## 10. Model Saving

In [ ]:
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save best non-naive model to disk
best_savable = results_df[results_df["object"].notna()].iloc[0]
best_model_object = best_savable["object"]
model_path = MODEL_DIR / "forecast_model.pkl"

joblib.dump(best_model_object, model_path)
print("Saved model:", model_path.resolve())

## 11. Key Insights

In [ ]:
print("Key Insights")
print("- Best model by RMSE:", results_df.iloc[0]["model"])
print("- Overall predictability: returns can be noisy; performance close to baseline suggests limited signal.")
print("- Limitations: short-term returns are volatile and may be influenced by external factors.")
print("- Improvements: add exogenous variables, tune hyperparameters, and evaluate rolling-window CV.")